# Proyectos de múltiples archivos en C++

**Curso:** Programación Avanzada  
**Sesión:** 19  
**Kernel:** xeus-cling (C++17)

---

Hasta ahora escribiste todo tu código en un solo archivo. Funciona bien para programas pequeños, pero en el mundo real los proyectos tienen decenas o cientos de archivos escritos por equipos distintos. En esta sesión vas a aprender cómo C++ organiza eso y cómo automatizar la construcción de un proyecto.

- **Archivos `.h`**: el *contrato* entre módulos — qué existe, no cómo funciona
- **Compilación por etapas**: `.cpp` → `.o` → ejecutable
- **Make**: no recompiles lo que no cambió
- **CMake**: describe tu proyecto una vez, compila en cualquier plataforma

La idea central: **separar la *declaración* (qué) de la *definición* (cómo)** es lo que permite que distintos archivos se conozcan sin depender unos de otros.

---

## Requisitos previos

Ejecuta esta celda primero — crea el directorio de trabajo y define un ayudante para capturar salida de terminal:

In [ ]:
#include <iostream>
#include <fstream>
#include <cstdio>
#include <string>
using namespace std;

// Crear directorio de trabajo para los demos
system("mkdir -p /tmp/s19");

// Ejecuta un comando de shell y muestra su salida en el notebook
auto shell = [](const string& cmd) {
    FILE* pipe = popen(cmd.c_str(), "r");
    char buf[512];
    while (fgets(buf, sizeof(buf), pipe))
        printf("%s", buf);
    pclose(pipe);
};

cout << "Listo. Directorio /tmp/s19 creado." << endl;

---

## Parte 1 — El problema: cuando todo cabe en un archivo

### 1.1 Un programa que crece

Imagina que empezaste con esto:

In [ ]:
{
    // Semana 1: todo en main.cpp, funciona perfecto
    void saludar(char* nombre) { cout << "Hola " << nombre << endl; }
    int  sumar(int a, int b)   { return a + b; }
    int  restar(int a, int b)  { return a - b; }

    char nombre[] = "Miguel";
    saludar(nombre);
    cout << "3 + 4 = " << sumar(3, 4) << endl;
}

Ahora imagina la semana 10: hay 30 funciones, 3 compañeros editando el mismo archivo, y cada cambio en `saludar` obliga a recompilar las 500 líneas de `main`. Los problemas:

| Problema | Consecuencia |
|----------|-------------|
| Un solo archivo enorme | Difícil de leer y mantener |
| Todos editan lo mismo | Conflictos de versión constantes |
| Recompilar todo siempre | Lento en proyectos grandes |
| No puedes reusar funciones | Copias y pegas en cada proyecto |

La solución: **separar el código en módulos**. Cada módulo tiene dos archivos:
- `modulo.h` — las *declaraciones* (qué funciones existen y qué tipo tienen)
- `modulo.cpp` — las *definiciones* (cómo funcionan)

---

## Parte 2 — Archivos de cabecera: el contrato entre módulos

### 2.1 Declaración vs. definición

En C++, hay una distinción fundamental:

```
Declaración  → le dice al compilador que algo EXISTE y cuál es su tipo
Definición   → le dice al compilador cómo funciona (ocupa código en memoria)
```

Puedes declarar la misma función mil veces (siempre que sea consistente), pero solo puedes *definirla* una vez. Eso es la **ODR** (*One Definition Rule*).

```
┌────────────────────────────┐   ┌───────────────────────────────────────────┐
│        funciones.h         │   │            funciones.cpp                  │
│  (declaraciones / contrato)│   │   (definiciones / implementación)         │
│────────────────────────────│   │───────────────────────────────────────────│
│                            │   │  #include "funciones.h"                   │
│  void saludar(char*);      │   │                                           │
│  int  sumar(int, int);     │   │  void saludar(char* n) {                  │
│  int  restar(int, int);    │   │      cout << "Hola " << n << endl;        │
│                            │   │  }                                        │
│                            │   │  int sumar(int a, int b) { return a+b; } │
│                            │   │  int restar(int a,int b) { return a-b; } │
└────────────────────────────┘   └───────────────────────────────────────────┘
            ↑                                        ↑
  main.cpp incluye esto            funciones.cpp incluye su propio .h
  para saber qué existe            para que el compilador verifique
                                   que la definición coincide
```

### 2.2 Guards de inclusión

¿Qué pasa si dos archivos incluyen el mismo `.h`?

```
main.cpp incluye funciones.h   ← primera vez
main.cpp incluye auxiliar.h   ← auxiliar.h también incluye funciones.h
```

El preprocesador copiaría `funciones.h` *dos veces* en `main.cpp` → error de redefinición. La solución son los **guards de inclusión**:

```cpp
// funciones.h
#ifndef FUNCIONES_H   // Si FUNCIONES_H no está definido...
#define FUNCIONES_H   // ...definirlo (primera vez que entra)

void saludar(char*);
int  sumar(int, int);
int  restar(int, int);

#endif                // Fin del bloque protegido
```

La segunda vez que el preprocesador ve `#include "funciones.h"`, ya existe `FUNCIONES_H`, así que salta todo el contenido.

> **Alternativa moderna:** `#pragma once` hace lo mismo en una línea y está soportado por todos los compiladores modernos.

In [ ]:
{
    // Las declaraciones pueden repetirse sin error (el compilador solo necesita saber que existe).
    // Las definiciones NO pueden repetirse — eso causaría el error que los guards evitan.
    void saludar(char*);   // primera declaracion
    void saludar(char*);   // segunda declaracion — OK

    cout << "Doble declaracion: sin error" << endl;

    // Descomentar esto SÍ daría error de redefinición:
    // void saludar(char* n) { cout << n; }
    // void saludar(char* n) { cout << n; }  // ERROR: redefinition
}

### 2.3 `<>` vs. `""`

| Sintaxis | Busca en... | Usar para... |
|----------|-------------|-------------|
| `#include <vector>` | Rutas del sistema (`/usr/include`, etc.) | Librería estándar y paquetes instalados |
| `#include "funciones.h"` | Directorio actual primero, luego sistema | Tus propios archivos |

Regla simple: si el archivo lo escribiste tú, usa `""`. Si viene con el compilador o una librería instalada, usa `<>`.

---

## Parte 3 — Compilación por etapas: `.cpp` → `.o` → ejecutable

### 3.1 Los dos pasos que `g++` hace en silencio

Cuando escribes `g++ main.cpp funciones.cpp -o programa`, en realidad pasan dos cosas:

```
Paso 1 — COMPILAR (cada .cpp independientemente → su .o)

  main.cpp      ──g++ -c──►  main.o       (código máquina de main)
  funciones.cpp ──g++ -c──►  funciones.o  (código máquina de funciones)

Paso 2 — ENLAZAR / LINK (todos los .o → ejecutable)

  main.o + funciones.o  ──g++──►  programa
```

El **compilador** traduce cada `.cpp` a código máquina de forma independiente.  
El **enlazador** (*linker*) conecta las llamadas entre archivos: cuando `main.o` llama a `saludar`, el linker busca en `funciones.o` dónde está esa función.

¿Por qué importa separar los pasos? Si cambias solo `funciones.cpp`, solo recompilas ese archivo → `funciones.o`. `main.o` ya existe y no cambia.

### 3.2 Demo: escribir archivos y compilar desde el notebook

Vamos a escribir los tres archivos al disco y compilarlos con `g++`:

In [ ]:
{
    // --- funciones.h ---
    ofstream fh("/tmp/s19/funciones.h");
    fh << "#ifndef FUNCIONES_H\n";
    fh << "#define FUNCIONES_H\n";
    fh << "\n";
    fh << "void saludar(char*);\n";
    fh << "int  sumar(int, int);\n";
    fh << "int  restar(int, int);\n";
    fh << "\n";
    fh << "#endif\n";
    fh.close();

    // --- funciones.cpp ---
    ofstream fc("/tmp/s19/funciones.cpp");
    fc << "#include <iostream>\n";
    fc << "#include \"funciones.h\"\n";
    fc << "using namespace std;\n";
    fc << "\n";
    fc << "void saludar(char* nombre) { cout << \"Hola \" << nombre << endl; }\n";
    fc << "int  sumar(int a, int b)   { return a + b; }\n";
    fc << "int  restar(int a, int b)  { return a - b; }\n";
    fc.close();

    // --- main.cpp ---
    ofstream fm("/tmp/s19/main.cpp");
    fm << "#include <iostream>\n";
    fm << "#include \"funciones.h\"\n";
    fm << "using namespace std;\n";
    fm << "\n";
    fm << "int main() {\n";
    fm << "    char nombre[] = \"Miguel\";\n";
    fm << "    saludar(nombre);\n";
    fm << "    cout << \"3 + 4 = \" << sumar(3, 4) << endl;\n";
    fm << "    cout << \"10 - 6 = \" << restar(10, 6) << endl;\n";
    fm << "}\n";
    fm.close();

    cout << "Archivos escritos en /tmp/s19/" << endl;
}

In [ ]:
// Paso 1: compilar cada .cpp a .o por separado
shell("cd /tmp/s19 && g++ -c funciones.cpp -o funciones.o 2>&1 && echo 'funciones.o OK'");
shell("cd /tmp/s19 && g++ -c main.cpp       -o main.o       2>&1 && echo 'main.o OK'");

In [ ]:
// ¿Qué hay en main.o? Símbolos definidos (T) y no resueltos (U)
shell("cd /tmp/s19 && nm --demangle main.o 2>/dev/null | grep -E 'saludar|sumar|restar|main'");

Los símbolos con `U` (*undefined*) son funciones que `main.o` llama pero cuya implementación está en otro `.o`. El linker resolverá esas referencias en el siguiente paso.

In [ ]:
// Paso 2: enlazar los .o → ejecutable
shell("cd /tmp/s19 && g++ main.o funciones.o -o programa 2>&1 && echo 'Enlace OK'");
shell("cd /tmp/s19 && ./programa");

In [ ]:
// Atajo: g++ acepta varios .cpp y hace los dos pasos internamente
shell("cd /tmp/s19 && g++ main.cpp funciones.cpp -o programa 2>&1 && ./programa");

---

## Parte 4 — Make: no recompiles lo que no cambió

### 4.1 El problema

Si tienes 50 archivos `.cpp` y cambias uno solo, no quieres recompilar los otros 49. **Make** resuelve esto: compara las fechas de modificación y solo recompila lo necesario.

### 4.2 Anatomía de un Makefile

Un Makefile es una lista de **reglas**. Cada regla tiene la forma:

```makefile
objetivo: dependencias
→TAB→ comando para construir el objetivo
```

> **El TAB es obligatorio.** Make exige que los comandos empiecen con una tabulación (no espacios). Es el error más común al escribir Makefiles.

In [ ]:
{
    ofstream mf("/tmp/s19/Makefile");
    mf << "# Makefile para el proyecto de funciones\n";
    mf << "\n";
    mf << "CXX      = g++\n";
    mf << "CXXFLAGS = -std=c++17 -Wall\n";
    mf << "TARGET   = programa\n";
    mf << "OBJS     = main.o funciones.o\n";
    mf << "\n";
    mf << "$(TARGET): $(OBJS)\n";
    mf << "\t$(CXX) $(OBJS) -o $(TARGET)\n";
    mf << "\n";
    mf << "main.o: main.cpp funciones.h\n";
    mf << "\t$(CXX) $(CXXFLAGS) -c main.cpp\n";
    mf << "\n";
    mf << "funciones.o: funciones.cpp funciones.h\n";
    mf << "\t$(CXX) $(CXXFLAGS) -c funciones.cpp\n";
    mf << "\n";
    mf << "clean:\n";
    mf << "\trm -f $(OBJS) $(TARGET)\n";
    mf.close();
    cout << "Makefile creado" << endl;
}

In [ ]:
// Primera compilación: construye todo desde cero
shell("cd /tmp/s19 && make clean 2>&1");
shell("cd /tmp/s19 && make 2>&1");

In [ ]:
// Segunda vez sin cambios: Make no hace nada
shell("cd /tmp/s19 && make 2>&1");

In [ ]:
// Simular un cambio en funciones.cpp (touch actualiza la fecha)
shell("touch /tmp/s19/funciones.cpp");
shell("cd /tmp/s19 && make 2>&1");
// Observa: solo recompila funciones.o, main.o no se toca

### 4.3 Variables y regla patrón

Para proyectos con muchos archivos, evitas repetir la misma regla con una **regla patrón**:

```makefile
# Cualquier .o se construye desde el .cpp con el mismo nombre
%.o: %.cpp
    $(CXX) $(CXXFLAGS) -c $< -o $@
```

Variables automáticas de Make:

| Variable | Significado |
|----------|-------------|
| `$@` | El objetivo actual (`funciones.o`) |
| `$<` | La primera dependencia (`funciones.cpp`) |
| `$^` | Todas las dependencias |
| `$(SRCS:.cpp=.o)` | Sustituye `.cpp` por `.o` en cada nombre |

---

## Parte 5 — CMake: un nivel más arriba

### 5.1 ¿Por qué CMake si ya tenemos Make?

Make funciona bien en Linux/Mac, pero en Windows la sintaxis es diferente. **CMake** soluciona esto: describes tu proyecto en `CMakeLists.txt` y CMake genera el sistema de compilación adecuado para cada plataforma.

```
CMakeLists.txt  ──cmake──►  Makefile       (Linux / Mac)
                         ►  .sln / .vcxproj (Visual Studio / Windows)
                         ►  build.ninja     (Ninja, multiplataforma)
```

### 5.2 Un `CMakeLists.txt` mínimo

In [ ]:
{
    ofstream cmake("/tmp/s19/CMakeLists.txt");
    cmake << "cmake_minimum_required(VERSION 3.10)\n";
    cmake << "\n";
    cmake << "project(Funciones VERSION 1.0)\n";
    cmake << "\n";
    cmake << "set(CMAKE_CXX_STANDARD 17)\n";
    cmake << "set(CMAKE_CXX_STANDARD_REQUIRED True)\n";
    cmake << "\n";
    cmake << "add_executable(programa\n";
    cmake << "    main.cpp\n";
    cmake << "    funciones.cpp\n";
    cmake << ")\n";
    cmake.close();
    cout << "CMakeLists.txt creado" << endl;
}

In [ ]:
// Flujo típico: build fuera del directorio fuente
shell("mkdir -p /tmp/s19/build");
shell("cd /tmp/s19/build && cmake .. 2>&1 | tail -6");

In [ ]:
// CMake generó un Makefile en build/; compilar y ejecutar
shell("cd /tmp/s19/build && make 2>&1");
shell("cd /tmp/s19/build && ./programa");

### 5.3 Ventajas del *out-of-source build*

CMake genera todos los archivos de compilación en `build/`, no en tu directorio fuente. Esto mantiene el repositorio limpio:

```
mi_proyecto/
├── CMakeLists.txt    ← solo esto va al repositorio
├── main.cpp
├── funciones.h
├── funciones.cpp
└── build/           ← generado por CMake, NO va al repositorio (.gitignore)
    ├── Makefile
    ├── main.o
    ├── funciones.o
    └── programa
```

### 5.4 Comparativa rápida

| Herramienta | Qué hace | Cuándo usarla |
|-------------|----------|---------------|
| `g++` directo | Compila y enlaza en un comando | Proyectos de 1–3 archivos |
| Make | Recompila solo lo que cambió | Proyectos medianos en Unix |
| CMake | Genera el sistema de compilación | Proyectos multiplataforma o con dependencias |
| Ninja, Bazel, Meson | Alternativas modernas a Make | Proyectos grandes / equipos grandes |

---

## Parte 6 — Proyecto real: sistema de personas

### 6.1 Estructura del proyecto

El directorio `materialInicial/sesion19/proyecto1/` tiene un proyecto con 7 archivos. Así está organizado:

```
proyecto1/
├── persona.h               ← struct Persona (solo datos, sin lógica)
├── auxiliar.h  / .cpp      ← generación aleatoria de nombres y apellidos
├── obtener_datos.h / .cpp  ← generar_personas() usando auxiliar
├── mate.h      / .cpp      ← cálculos numéricos (promedios de edad y salario)
├── buscar.h    / .cpp      ← búsquedas y filtros sobre arreglos de Persona
└── main.cpp                ← orquesta todo
```

El grafo de dependencias:

```
main.cpp
  ├── persona.h          ← struct Persona
  ├── buscar.h  → persona.h
  ├── mate.h    → persona.h
  └── obtener_datos.h → persona.h
                      → auxiliar.h
```

`persona.h` está en el centro porque todos los módulos trabajan con `Persona`. Los guards de inclusión evitan que se procese varias veces.

In [ ]:
// Mostrar persona.h — el centro del proyecto
shell("find /home -name 'persona.h' -path '*/sesion19/*' 2>/dev/null | head -1 | xargs cat");

In [ ]:
// Mostrar mate.h — ejemplo de módulo que depende de persona.h
shell("find /home -name 'mate.h' -path '*/sesion19/*' 2>/dev/null | head -1 | xargs cat");

**¿Por qué `mate.h` incluye `persona.h`?**  
Porque declara funciones que reciben un arreglo de `Persona`. El compilador necesita saber qué es `Persona` para entender la firma `promedio_edades(Persona[], int)`.

In [ ]:
// CMakeLists.txt para el proyecto1
{
    string proj = "";
    FILE* f = popen("find /home -path '*/sesion19/proyecto1' -type d 2>/dev/null | head -1", "r");
    char buf[512];
    if (fgets(buf, sizeof(buf), f)) {
        proj = string(buf);
        while (!proj.empty() && (proj.back() == '\n' || proj.back() == ' ')) proj.pop_back();
    }
    pclose(f);

    if (proj.empty()) {
        cout << "No encontre el directorio proyecto1" << endl;
    } else {
        ofstream cmake(proj + "/CMakeLists.txt");
        cmake << "cmake_minimum_required(VERSION 3.10)\n";
        cmake << "project(SistemaPersonas VERSION 1.0)\n";
        cmake << "set(CMAKE_CXX_STANDARD 17)\n";
        cmake << "set(CMAKE_CXX_STANDARD_REQUIRED True)\n";
        cmake << "\n";
        cmake << "add_executable(programa\n";
        cmake << "    main.cpp\n";
        cmake << "    auxiliar.cpp\n";
        cmake << "    obtener_datos.cpp\n";
        cmake << "    mate.cpp\n";
        cmake << "    buscar.cpp\n";
        cmake << ")\n";
        cmake.close();
        cout << "CMakeLists.txt escrito en: " << proj << endl;
    }
}

---

## Ejercicios

### Ejercicio 1 — Agregar un módulo

Agrega una función `int multiplicar(int a, int b)` al proyecto simple (`/tmp/s19/`). Debes:
1. Declarar la función en `funciones.h`
2. Definirla en `funciones.cpp`
3. Llamarla desde `main.cpp`
4. Recompilar con `make` y verificar que solo se recompila `funciones.o`

In [ ]:
// Tu solucion aqui

### Ejercicio 2 — ¿Qué pasa sin los guards?

Modifica `funciones.h` en `/tmp/s19/` para quitar el `#ifndef`/`#define`/`#endif`. Luego escribe un `main2.cpp` que incluya `funciones.h` dos veces e intenta compilarlo. Explica el error.

In [ ]:
// Tu solucion aqui

### Ejercicio 3 — Nuevo módulo para proyecto1

Crea `estadisticas.h` / `estadisticas.cpp` con la función:
```cpp
double salario_maximo(Persona arr[], int n);
```
Incluye el guard de inclusión correcto, agrega el `.cpp` al `CMakeLists.txt` y llama la función desde `main.cpp`.

In [ ]:
// Tu solucion aqui

---

| Concepto | Archivo / Herramienta | Qué contiene / hace |
|----------|-----------------------|---------------------|
| Declaración | `modulo.h` | Firmas de funciones y structs; no código ejecutable |
| Definición | `modulo.cpp` | Implementación; incluye su propio `.h` |
| Guard de inclusión | `#ifndef` / `#define` / `#endif` | Evita procesar el `.h` dos veces |
| Compilar a `.o` | `g++ -c archivo.cpp` | Traduce un `.cpp` a código máquina sin enlazar |
| Enlazar | `g++ a.o b.o -o prog` | Conecta referencias entre módulos → ejecutable |
| Make | `Makefile` | Recompila solo los archivos modificados |
| CMake | `CMakeLists.txt` | Genera el sistema de compilación multiplataforma |

---
*Programación Avanzada — Universidad Panamericana*